# VRSBench Image Captioning — Moondream2 + LoRA + Length-Penalty Loss
Same architecture and overall pipeline as the SmolVLM2 run: LoRA finetune on the **full** VRSBench
training set, a length-penalty loss term pulling generated captions toward the dataset's median
length (53 words), a 9hr wall-clock training cap, and the same metric suite / 50-sample official
val eval at the end.

Moondream2 does **not** use a standard chat-template + `Trainer` pipeline like SmolVLM — it's a
custom `trust_remote_code=True` architecture (a Phi-based text decoder + a separate vision encoder,
loaded via `AutoModelForCausalLM`) that builds its own `[BOS] + image_embeds + question_tokens +
answer_tokens` input sequence and doesn't expose a HF-`Trainer`-compatible `forward(labels=...)`
in the way SmolVLM's `AutoModelForImageTextToText` does. So Section 9 uses a manual training loop
instead of `Trainer`, built on the pattern from the **official moondream finetuning notebook**
(`m87-labs/moondream`, PR #227/#250, `finetune/finetune_text.py` — LoRA/full finetune of
`text_model` only, vision encoder frozen) and cross-checked against public moondream2 finetunes on
the Hub (e.g. `zesquirrelnator/moondream2-finetuneV2`, `gokaygokay/moondream-prompt`) for the
`encode_image` / `answer_question` / prompt-format conventions. Everything else — dataset, LoRA
rank/alpha, length-penalty formula, time-cap callback logic, metrics, eval split — mirrors the
SmolVLM notebook as closely as the architecture allows.

## 1. Setup — installs, imports, data + model download (everything up front)

In [1]:
!pip install -q --no-deps "transformers==4.43.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 58.9 MB/s eta 0:00:00


In [2]:
!pip install -q accelerate peft bitsandbytes einops timm num2words
!pip install -q sentence-transformers nltk rouge-score pycocoevalcap


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 17.0 MB/s eta 0:00:00


In [3]:
import numpy as np
import transformers
import torch

print("NumPy:", np.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

NumPy: 2.0.2
Transformers: 4.43.4
PyTorch: 2.10.0+cu128
CUDA: True


In [4]:
import os
import json
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
print("Device:", DEVICE, "| dtype:", DTYPE)

RUN_START = time.time()
def elapsed_min():
    return (time.time() - RUN_START) / 60

Device: cuda | dtype: torch.bfloat16


In [5]:
# --- Dataset paths (mounted Kaggle input, no download needed) ---
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train_caption.json")
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")
VAL_IMG_DIR = os.path.join(DATA_DIR, "Images_val", "Images_val")
EVAL_CAP_JSON = os.path.join(DATA_DIR, "VRSBench_EVAL_Cap.json")

# --- Output directory (Kaggle persists /kaggle/working/ as notebook output) ---
OUTPUT_DIR = "/kaggle/working/moondream2_vrsbench_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
# --- Model + tokenizer download (pulled once here, used by every later cell) ---
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_NAME = "vikhyatk/moondream2"
MD_REVISION = "2024-08-26"   # pin a revision — moondream2 weights/API have changed across revisions

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MD_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MD_REVISION,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    attn_implementation="eager",   # flash-attn optional; eager is the safe default for finetuning
).to(DEVICE)

# Moondream2 is two sub-modules under one AutoModelForCausalLM: a frozen-by-default vision encoder
# (model.vision_encoder) and a Phi-based text decoder (model.text_model). Only text_model gets LoRA
# here, matching the official finetuning notebook (vision encoder stays frozen).
for p in model.vision_encoder.parameters():
    p.requires_grad = False

print(f"Model + tokenizer downloaded. Elapsed: {elapsed_min():.1f} min")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

...

vision_encoder.py: 0.00B [00:00, ?B/s]

modeling_phi.py: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.74G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model + tokenizer downloaded. Elapsed: 0.7 min


## 2. Build DataFrame from train captions

In [7]:
DATA_DIR = "/kaggle/input/datasets/ayaanmustafa/vrs-bench/vrsbench_data"
ANNOT_FILE = os.path.join(DATA_DIR, "VRSBench_train.json")  # update filename if different
IMG_DIR = os.path.join(DATA_DIR, "Images_train", "Images_train")

with open(ANNOT_FILE, "r") as f:
    raw = json.load(f)

caption_rows = []

for item in raw:
    conversations = item["conversations"]

    if "[caption]" in conversations[0]["value"].lower():
        caption_rows.append({
            "id": item["id"],
            "image": item["image"],
            "caption": conversations[1]["value"]
        })

df = pd.DataFrame(caption_rows)
# Normalize columns: expect a filename column and a caption column
# (adjust the renames below to match the printed column names above)
rename_map = {}
for c in df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        rename_map[c] = "filename"
    elif lc in ("image_path", "path"):
        rename_map[c] = "filename"
    elif "caption" in lc:
        rename_map[c] = "caption"

df = df.rename(columns=rename_map)
df["image_path"] = df["filename"].apply(lambda x: os.path.join(IMG_DIR, os.path.basename(str(x))))

df = df[["image_path", "caption"]].dropna().reset_index(drop=True)
print(df.shape)
df.head()

(20264, 2)


,image_path,caption
0,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a r..."
1,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,This aerial image from Google Earth shows a de...
2,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."
3,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, shows a f..."
4,/kaggle/input/datasets/ayaanmustafa/vrs-bench/...,"The image, sourced from GoogleEarth, showcases..."


## 3. Train set — full dataset

In [8]:
from sklearn.model_selection import train_test_split

train_df, _ = train_test_split(df, test_size=0.1, random_state=SEED)

# Full dataset — no subsampling. Training is wall-clock capped (Section 9), so the full pool
# just means more of it gets seen; the time callback still governs when training stops.
train_df = train_df.reset_index(drop=True)

print("Train:", len(train_df))

Train: 18237


In [9]:
!pip install -q --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 62.7 MB/s eta 0:00:00


## 4. LoRA setup
Rank 16 / alpha 16 — same as the SmolVLM run — applied to `text_model` only (vision encoder frozen,
matching the official moondream finetuning notebook). Moondream2's text decoder is a Phi-arch model,
so the attention/MLP module names differ from SmolVLM's Llama-style `q_proj`/`k_proj`/... naming:
Phi fuses QKV into `Wqkv` and uses `fc1`/`fc2` for the MLP.

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["Wqkv", "out_proj", "fc1", "fc2"],  # Phi-style module names used by moondream2's text_model
    task_type="CAUSAL_LM",
)
model.text_model = get_peft_model(model.text_model, lora_config)
model.text_model.print_trainable_parameters()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


trainable params: 12,582,912 || all params: 1,430,853,632 || trainable%: 0.8794


## 5. Metric-aware prompt
Median caption length in VRSBench is 53 words — stated explicitly in the prompt, same as the
SmolVLM run. Moondream2 doesn't use a chat template; its convention (and the one used by public
finetunes like `gokaygokay/moondream-prompt`) is a plain `"\n\nQuestion: {q}\n\nAnswer:"` /
`" {a}<|endoftext|>"` format around a single embedded image, with no image few-shot support in the
stock API (each call takes exactly one image). So the few-shot-with-images approach from the
SmolVLM prompt is dropped here — the 53-word target instruction goes directly in the question text
instead.

In [11]:
MEDIAN_WORDS = 53
ANSWER_EOS = "<|endoftext|>"

QUESTION_TEXT = (
    "Describe this satellite/aerial image in ONE concise paragraph of about "
    f"{MEDIAN_WORDS} words. Mention key objects, their approximate spatial layout "
    "(e.g. top-left, bottom-right, center), and scene context. Do not hedge, apologize, "
    "or comment on image quality. Do not repeat phrases."
)

## 6. PyTorch Dataset
Same loose safety cap (90 words) as the SmolVLM run — not the primary length-control mechanism,
just a guard against one outlier caption dominating a batch. The length-penalty loss (Section 8)
does the actual length-shaping via gradient signal.

In [12]:
from torch.utils.data import Dataset

class VRSBenchMoondreamDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        caption = row["caption"]
        words = caption.split()
        if len(words) > 90:   # loose safety cap only, not the primary length-control mechanism
            caption = " ".join(words[:90])
        return {"image": image, "question": QUESTION_TEXT, "answer": caption}


train_dataset = VRSBenchMoondreamDataset(train_df)
print(len(train_dataset))

18237


## 7. Collator
Follows the official moondream finetuning notebook's sequence layout: `[BOS] + IMG_TOKENS(-100) +
question_tokens(-100) + answer_tokens(labeled) + EOS`. Images go through `model.vision_encoder`'s own
preprocessing, and the 729 image-embedding slots are masked out of the loss (`-100`) since they
aren't token ids.

In [13]:
IMG_TOKENS = 729

def collate_fn(examples):
    # Moondream2 preprocess() accepts ONE PIL image at a time.
    images = torch.stack([
        model.vision_encoder.preprocess(ex["image"])
        for ex in examples
    ]).to(DEVICE, dtype=DTYPE)

    labels_acc, tokens_acc = [], []

    for ex in examples:
        # BOS
        toks = [tokenizer.bos_token_id]

        # BOS + 729 image embedding slots are not language-model targets
        labs = [-100] * (IMG_TOKENS + 1)

        # Question
        q_ids = tokenizer(
            f"\n\nQuestion: {ex['question']}\n\nAnswer:",
            add_special_tokens=False
        ).input_ids

        toks.extend(q_ids)
        labs.extend([-100] * len(q_ids))

        # Answer
        a_ids = tokenizer(
            f" {ex['answer']}{ANSWER_EOS}",
            add_special_tokens=False
        ).input_ids

        toks.extend(a_ids)
        labs.extend(a_ids)

        tokens_acc.append(toks)
        labels_acc.append(labs)

    # Pad batch
    max_len = max(len(x) for x in labels_acc)

    attn_mask_acc = []

    for i in range(len(examples)):
        pad = max_len - len(labels_acc[i])

        labels_acc[i].extend([-100] * pad)
        tokens_acc[i].extend([tokenizer.eos_token_id] * pad)

        attn_mask_acc.append(
            [1] * (max_len - pad) + [0] * pad
        )

    return {
        "images": images,
        "tokens": torch.tensor(
            tokens_acc,
            dtype=torch.long,
            device=DEVICE
        ),
        "labels": torch.tensor(
            labels_acc,
            dtype=torch.long,
            device=DEVICE
        ),
        "attn_mask": torch.tensor(
            attn_mask_acc,
            dtype=torch.bool,
            device=DEVICE
        ),
    }

## 8. Length-penalty loss — penalizes deviation from the median (53 words)
Same idea as the SmolVLM run: an explicit auxiliary term added on top of the model's own
cross-entropy loss, penalizing the squared deviation of each sample's *answer* token count from a
target token count corresponding to `MEDIAN_WORDS`. `LENGTH_PENALTY_LAMBDA` is set to **0.06** (a
firmer pull than the earlier 0.02 run). Since moondream2's `text_model.forward` doesn't route
through a `Trainer.compute_loss` hook, this is computed directly inside the manual training loop
(Section 9) rather than in a `Trainer` subclass.

In [14]:
SAMPLE_TEXT = " ".join(["word"] * MEDIAN_WORDS)
TARGET_TOKEN_COUNT = len(tokenizer(SAMPLE_TEXT, add_special_tokens=False)["input_ids"])
print("Target token count for median length:", TARGET_TOKEN_COUNT)

LENGTH_PENALTY_LAMBDA = 0.05   # weight of the length term relative to CE loss — a firmer pull than before

def length_penalty(labels, response_lengths):
    deviation = (response_lengths - TARGET_TOKEN_COUNT) ** 2
    return deviation.mean() / (TARGET_TOKEN_COUNT ** 2)   # normalized so it stays a small, stable term


def compute_loss(batch):
    images, tokens, labels, attn_mask = batch["images"], batch["tokens"], batch["labels"], batch["attn_mask"]

    with torch.no_grad():
        img_embs = model.vision_encoder(images)

    tok_embs = model.text_model.get_input_embeddings()(tokens)
    # Splice image embeddings in right after the BOS token, matching moondream2's own encode_image
    # + answer_question conventions (and the official finetuning notebook's collator).
    inputs_embeds = torch.cat((tok_embs[:, 0:1, :], img_embs, tok_embs[:, 1:, :]), dim=1)

    outputs = model.text_model(
        inputs_embeds=inputs_embeds,
        labels=labels,
        attention_mask=attn_mask,
    )
    ce_loss = outputs.loss

    response_lengths = (labels != -100).sum(dim=-1).float()
    len_loss = length_penalty(labels, response_lengths)

    loss = ce_loss + LENGTH_PENALTY_LAMBDA * len_loss
    return loss, ce_loss, len_loss, response_lengths

Target token count for median length: 53


In [15]:
from sentence_transformers import SentenceTransformer

bert_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(DEVICE))

def get_ngrams(tokens, n):
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]

def bert_bleu4(candidate, reference, alpha=0.5):
    c_tokens = candidate.lower().split()
    r_tokens = reference.lower().split()
    c_ngrams_by_n = [get_ngrams(c_tokens, n) for n in range(1, 5)]
    r_ngrams_by_n = [get_ngrams(r_tokens, n) for n in range(1, 5)]
    all_texts = [g for grams in c_ngrams_by_n + r_ngrams_by_n for g in grams]
    if not all_texts:
        return 0.0
    embs = bert_model.encode(all_texts, convert_to_tensor=True, show_progress_bar=False, batch_size=256)
    embs = F.normalize(embs, dim=-1)
    idx = 0
    c_embs_by_n, r_embs_by_n = [], []
    for grams in c_ngrams_by_n:
        c_embs_by_n.append(embs[idx: idx + len(grams)]); idx += len(grams)
    for grams in r_ngrams_by_n:
        r_embs_by_n.append(embs[idx: idx + len(grams)]); idx += len(grams)
    log_p_sum = 0.0
    for n in range(4):
        C_emb, R_emb = c_embs_by_n[n], r_embs_by_n[n]
        if len(C_emb) == 0 or len(R_emb) == 0:
            p_n = 1e-9
        else:
            sims = torch.mm(R_emb, C_emb.T)
            p_n = max(sims.max(dim=1).values.mean().item(), 1e-9)
        log_p_sum += np.log(p_n)
    L_c, L_r = len(c_tokens), max(len(r_tokens), 1)
    lp = np.exp(-alpha * abs(L_c - L_r) / L_r)
    return lp * np.exp(log_p_sum / 4)

def corpus_bert_bleu4(candidates, references):
    return float(np.mean([bert_bleu4(c, r) for c, r in zip(candidates, references)]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider

smoothie = SmoothingFunction().method4
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def compute_bleu_n(candidates, references, n):
    weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
    scores = []
    for c, r in zip(candidates, references):
        c_tok, r_tok = c.lower().split(), [r.lower().split()]
        scores.append(sentence_bleu(r_tok, c_tok, weights=weights, smoothing_function=smoothie))
    return float(np.mean(scores))

def compute_meteor(candidates, references):
    return float(np.mean([meteor_score([r.lower().split()], c.lower().split()) for c, r in zip(candidates, references)]))

def compute_rouge_l(candidates, references):
    return float(np.mean([rouge.score(r, c)["rougeL"].fmeasure for c, r in zip(candidates, references)]))

def compute_cider(candidates, references):
    gts = {i: [r] for i, r in enumerate(references)}
    res = {i: [c] for i, c in enumerate(candidates)}
    score, _ = Cider().compute_score(gts, res)
    return float(score)

def compute_avg_len(candidates):
    return float(np.mean([len(c.split()) for c in candidates]))

def compute_all_metrics(candidates, references):
    return {
        "BLEU-1": compute_bleu_n(candidates, references, 1),
        "BLEU-2": compute_bleu_n(candidates, references, 2),
        "BLEU-3": compute_bleu_n(candidates, references, 3),
        "BLEU-4": compute_bleu_n(candidates, references, 4),
        "METEOR": compute_meteor(candidates, references),
        "ROUGE_L": compute_rouge_l(candidates, references),
        "CIDEr": compute_cider(candidates, references),
        "Avg_L": compute_avg_len(candidates),
        "BERT-BLEU4": corpus_bert_bleu4(candidates, references),
    }

## 9. Training — wall-clock capped to 9hr
No `Trainer` here (see the note in Section 1) — a manual loop over `DataLoader` batches, with
gradient accumulation to match the SmolVLM run's effective batch size (1 x 16 = 16), a cosine LR
schedule with warmup, and the same wall-clock stopping behavior as `TimeLimitCallback`: training
runs until 9 hours elapse or the data is exhausted across epochs, whichever comes first, rather than
guessing an epoch count ahead of time.

In [17]:
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
import bitsandbytes as bnb

TARGET_TRAIN_MINUTES = 600   # 9hr wall-clock cap on training
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16        # effective batch still 16, matching the SmolVLM run
LEARNING_RATE = 2e-4
WARMUP_STEPS = 50
NUM_EPOCHS_UPPER_BOUND = 50  # high upper bound — the time cap below stops training first
LOGGING_STEPS = 20
SAVE_STEPS = 100
CHECKPOINT_DIR = "/kaggle/working/moondream2_vrsbench_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_loader = DataLoader(
    train_dataset, batch_size=PER_DEVICE_BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)

trainable_params = [p for p in model.text_model.parameters() if p.requires_grad]
optimizer = bnb.optim.AdamW8bit(trainable_params, lr=LEARNING_RATE)

steps_per_epoch = max(1, len(train_loader) // GRAD_ACCUM_STEPS)
total_steps_upper_bound = steps_per_epoch * NUM_EPOCHS_UPPER_BOUND
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps_upper_bound
)

model.text_model.train()

# Required because we manually pass inputs_embeds into the checkpointed model.
model.text_model.enable_input_require_grads()

# Gradient checkpointing
model.text_model.gradient_checkpointing_enable()

# Required when using gradient checkpointing
if hasattr(model.text_model, "config"):
    model.text_model.config.use_cache = False

train_start = time.time()
target_seconds = TARGET_TRAIN_MINUTES * 60
global_step = 0
micro_step = 0
stop_training = False

for epoch in range(NUM_EPOCHS_UPPER_BOUND):
    if stop_training:
        break
    for batch in train_loader:
        loss, ce_loss, len_loss, response_lengths = compute_loss(batch)
        (loss / GRAD_ACCUM_STEPS).backward()
        micro_step += 1

        if micro_step % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % LOGGING_STEPS == 0:
                print(f"  [epoch {epoch} step {global_step}] ce={ce_loss.item():.4f} "
                      f"len_penalty={len_loss.item():.4f} "
                      f"avg_response_tokens={response_lengths.mean().item():.1f}")

            if global_step % SAVE_STEPS == 0:
                ckpt_path = os.path.join(CHECKPOINT_DIR, f"step-{global_step}")
                model.text_model.save_pretrained(ckpt_path)  # LoRA adapter only, periodic safety save

        elapsed = time.time() - train_start
        if elapsed >= target_seconds:
            print(f"\n[TimeLimit] Reached {elapsed/60:.1f} min — stopping training gracefully "
                  f"(completed {global_step} optimizer steps, epoch {epoch}).")
            stop_training = True
            break

print(f"Training took {elapsed_min():.1f} min total (incl. setup)")

  [epoch 0 step 20] ce=2.5505 len_penalty=0.0032 avg_response_tokens=56.0
  [epoch 0 step 40] ce=1.8454 len_penalty=0.1723 avg_response_tokens=75.0
  [epoch 0 step 60] ce=1.4680 len_penalty=0.4614 avg_response_tokens=89.0
  [epoch 0 step 80] ce=1.3140 len_penalty=0.0356 avg_response_tokens=43.0
  [epoch 0 step 100] ce=1.5330 len_penalty=0.5415 avg_response_tokens=92.0


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in  - will assume that the vocabulary was not modified.
  warnings.warn(


  [epoch 0 step 120] ce=1.6493 len_penalty=0.0174 avg_response_tokens=60.0
  [epoch 0 step 140] ce=1.4128 len_penalty=0.3877 avg_response_tokens=86.0
  [epoch 0 step 160] ce=1.2185 len_penalty=0.0513 avg_response_tokens=65.0
  [epoch 0 step 180] ce=1.5930 len_penalty=0.0513 avg_response_tokens=41.0
  [epoch 0 step 200] ce=1.6483 len_penalty=0.0000 avg_response_tokens=53.0
  [epoch 0 step 220] ce=1.4189 len_penalty=0.0431 avg_response_tokens=64.0
  [epoch 0 step 240] ce=1.3317 len_penalty=0.1153 avg_response_tokens=71.0
  [epoch 0 step 260] ce=1.5596 len_penalty=0.1723 avg_response_tokens=75.0
  [epoch 0 step 280] ce=1.4677 len_penalty=0.1029 avg_response_tokens=70.0
  [epoch 0 step 300] ce=1.3947 len_penalty=0.0174 avg_response_tokens=46.0
  [epoch 0 step 320] ce=1.4939 len_penalty=0.0004 avg_response_tokens=52.0
  [epoch 0 step 340] ce=1.4929 len_penalty=0.2407 avg_response_tokens=79.0
  [epoch 0 step 360] ce=1.3233 len_penalty=1.3247 avg_response_tokens=114.0
  [epoch 0 step 380] ce=

## 10. Save final model to the output directory

In [18]:
model.text_model.save_pretrained(OUTPUT_DIR)   # LoRA adapter weights
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved final LoRA adapter + tokenizer to:", OUTPUT_DIR)
!ls {OUTPUT_DIR}

Saved final LoRA adapter + tokenizer to: /kaggle/working/moondream2_vrsbench_final
adapter_config.json	   merges.txt		    tokenizer_config.json
adapter_model.safetensors  README.md		    tokenizer.json
added_tokens.json	   special_tokens_map.json  vocab.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


## 11. Metric suite

## 12. Load official val split, 50-sample eval

In [19]:
with open(EVAL_CAP_JSON, "r") as f:
    eval_raw = json.load(f)

eval_df = pd.DataFrame(eval_raw)
if "type" in eval_df.columns:
    eval_df = eval_df[eval_df["type"] == "caption"].reset_index(drop=True)

eval_rename = {}
for c in eval_df.columns:
    lc = c.lower()
    if "image" in lc and "path" not in lc:
        eval_rename[c] = "filename"
    elif lc in ("image_path", "path"):
        eval_rename[c] = "filename"
    elif "caption" in lc or lc in ("ground_truth", "answer"):
        eval_rename[c] = "caption"

eval_df = eval_df.rename(columns=eval_rename)
eval_df["image_path"] = eval_df["filename"].apply(lambda x: os.path.join(VAL_IMG_DIR, os.path.basename(str(x))))
eval_df = eval_df[["image_path", "caption"]].dropna().reset_index(drop=True)

EVAL_SUBSET_SIZE = 50
eval_subset_df = eval_df.sample(n=min(EVAL_SUBSET_SIZE, len(eval_df)), random_state=SEED).reset_index(drop=True)
print("Eval samples used:", len(eval_subset_df))

Eval samples used: 50


## 13. Inference
Uses moondream2's own `encode_image` + `answer_question` API (the same one used by public
finetunes like `gokaygokay/moondream-prompt`), so the LoRA adapter merged into `text_model` is
exercised through the model's native generation path rather than a hand-rolled `generate()` call.
No hard word-count truncation — same rationale as the SmolVLM run: seeing what the length-penalty
loss learned on its own.

In [20]:
model.text_model.eval()

@torch.no_grad()
def generate_caption(image, max_new_tokens=120):
    image_embeds = model.encode_image(image)
    answer = model.answer_question(
        image_embeds,
        QUESTION_TEXT,
        tokenizer=tokenizer,
        max_new_tokens=max_new_tokens,
    )
    return answer.strip()


def validate(eval_df):
    preds, refs = [], []
    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="val"):
        image = Image.open(row["image_path"]).convert("RGB")
        pred = generate_caption(image)
        preds.append(pred)
        refs.append(row["caption"])
    metrics = compute_all_metrics(preds, refs)
    return metrics, preds, refs


metrics, preds, refs = validate(eval_subset_df)
print(metrics)
print(f"Avg predicted length: {np.mean([len(p.split()) for p in preds]):.1f} words (target median: {MEDIAN_WORDS})")
print(f"Total run time: {elapsed_min():.1f} min")

val:   0%|          | 0/50 [00:00<?, ?it/s]

{'BLEU-1': 0.3596942869791134, 'BLEU-2': 0.22760985325055752, 'BLEU-3': 0.14819218220553193, 'BLEU-4': 0.0996497740108629, 'METEOR': 0.32747994693489174, 'ROUGE_L': 0.3779817030264426, 'CIDEr': 0.2828607337399696, 'Avg_L': 42.24, 'BERT-BLEU4': 0.5810205569879046}
Avg predicted length: 42.2 words (target median: 53)
Total run time: 604.6 min


## 14. Sample predictions

In [21]:
for p, r in list(zip(preds, refs))[:5]:
    print(f"PRED ({len(p.split())}w):", p)
    print(f"REF  ({len(r.split())}w):", r)
    print("-" * 60)

PRED (31w): The high-resolution image from GoogleEarth shows a large ship docked at a pier. The ship is positioned in the bottom-right part of the image, extending from the pier into the water.
REF  (24w): The image from GoogleEarth shows a docking area with a ship. The ship has a white and green color scheme and is very long.
------------------------------------------------------------
PRED (29w): The high-resolution image from GoogleEarth shows a recreational area with a basketball court located at the bottom of the frame. The court is surrounded by various structures and trees.
REF  (35w): This aerial image from GoogleEarth features a variety of elements including a prominent basketball court. The basketball court is prominently placed in the lower central part of the image, paved in gray with white markings.
------------------------------------------------------------
PRED (59w): The high-resolution image from GoogleEarth shows a section of a harbor area adjacent to a bridge. There